## 1. Импорт библиотек и чтение данных

In [1]:
from pathlib import Path
import json
import sys
import joblib
import pandas as pd

base = Path.cwd()
if not (base / "model.joblib").exists():
    base = base / "hw_10"
assert (base / "model.joblib").exists(), "Откройте ноутбук из папки hw_10 или корня репозитория"

info = json.loads((base / "model_info.json").read_text(encoding="utf-8"))
dataset = pd.read_csv(base / "realty_train.csv")
print("Строк в учебном датасете:", len(dataset))
print("Признаки модели:", info["features"])
display(dataset.head())

Строк в учебном датасете: 98822
Признаки модели: ['city', 'total_square', 'rooms', 'floor']


,price,city,total_square,rooms,floor
0,63000000,Москва,137.0,3.0,6.0
1,3250000,Москва,16.7,0.0,1.0
2,16004680,Москва,76.0,3.0,6.0
3,7841776,Москва,24.0,1.0,10.0
4,120000000,Москва,126.0,3.0,16.0


## 2. Готовая модель и воспроизводимость

In [2]:
model = joblib.load(base / "model.joblib")
print("Строк после подготовки:", info["n_records"])
print("Обучение:", info["n_train"], "Тест:", info["n_test"])
print("MAE:", f"{info['mae_rub']:,}".replace(",", " "), "руб.")
print("R²:", info["r2"])
print("random_state:", info["random_state"])


Строк после подготовки: 88114
Обучение: 70491 Тест: 17623
MAE: 6 163 536 руб.
R²: 0.7802
random_state: 42


## 3. Функция прогнозирования обученной моделью

In [3]:
def predict_price(city: str, total_square: float, rooms: int, floor: int) -> float:
    features = pd.DataFrame([{
        "city": city,
        "total_square": total_square,
        "rooms": rooms,
        "floor": floor,
    }])
    return round(max(0.0, float(model.predict(features)[0])), 2)

price = predict_price("Москва", 60.0, 2, 5)
print(f"Прогноз: {price:,.2f} руб.".replace(",", " "))

Прогноз: 20 113 782.87 руб.


## 4. Создание REST API

В проекте находится файл [`app.py`](app.py), содержащий:

- `GET /predict_get` — параметры из URL;
- `POST /predict_post` — параметры из JSON-тела;
- `GET /health` — проверка загрузки модели.


## 5. Проверка GET /health

In [ ]:
sys.path.insert(0, str(base.resolve()))
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)
health = client.get("/health")
print("Статус:", health.status_code)
print("Ответ:", health.json())
assert health.status_code == 200 and health.json()["model_loaded"]

Статус: 200
Ответ: {'status': 'ok', 'model_loaded': True}


## 6. Проверка GET /predict_get

In [5]:
example = {"city": "Москва", "total_square": 60.0, "rooms": 2, "floor": 5}
response_get = client.get("/predict_get", params=example)
print("HTTP:", response_get.status_code)
print(json.dumps(response_get.json(), ensure_ascii=False, indent=2))
assert response_get.status_code == 200
assert response_get.json()["predicted_price_rub"] > 0

HTTP: 200
{
  "predicted_price_rub": 20113782.87,
  "price_per_sqm_rub": 335229.71,
  "currency": "RUB",
  "features": {
    "city": "Москва",
    "total_square": 60.0,
    "rooms": 2,
    "floor": 5
  },
  "note": "Учебный прогноз по объявлениям, не оценочное заключение"
}


## 7. Проверка POST /predict_post

In [6]:
response_post = client.post("/predict_post", json=example)
print("HTTP:", response_post.status_code)
print(json.dumps(response_post.json(), ensure_ascii=False, indent=2))
assert response_post.status_code == 200
assert response_get.json()["predicted_price_rub"] == response_post.json()["predicted_price_rub"]
print("Одинаковые параметры → одинаковая цена при GET и POST")

HTTP: 200
{
  "predicted_price_rub": 20113782.87,
  "price_per_sqm_rub": 335229.71,
  "currency": "RUB",
  "features": {
    "city": "Москва",
    "total_square": 60.0,
    "rooms": 2,
    "floor": 5
  },
  "note": "Учебный прогноз по объявлениям, не оценочное заключение"
}
Одинаковые параметры → одинаковая цена при GET и POST


## 8. Проверка некорректных данных

In [7]:
invalid = client.get("/predict_get", params={**example, "total_square": -1})
unknown = client.post("/predict_post", json={**example, "city": "НесуществующийГород"})
print("Отрицательная площадь:", invalid.status_code)
print("Неизвестный город:", unknown.status_code)
assert invalid.status_code == 422
assert unknown.status_code == 422

Отрицательная площадь: 422
Неизвестный город: 422
